# Práctica: Predicción de Salarios con Regresiones Penalizadas

## Contexto

Utilizamos datos de la **Current Population Survey (CPS)** de Estados Unidos para modelar el logaritmo del salario por hora (`lwage`) en función de características individuales: educación, experiencia, ocupación, industria y región.

El ejercicio tiene dos partes con una motivación clara:

- **Parte 1 — Modelo básico ($p < n$):** Pocas variables, OLS es viable. Sirve como *benchmark*.
- **Parte 2 — Modelo con interacciones ($p \gg n$):** Al agregar todas las interacciones entre experiencia y el resto de covariables, el número de columnas explota y OLS falla. Aquí entran Lasso, Ridge y Elastic Net.

La pregunta central es: **¿cuánto mejora la predicción al pasar de un modelo simple a uno con cientos de interacciones, y qué método penalizado lo aprovecha mejor?**

---

## Fórmulas de los métodos

Todos minimizan la misma función objetivo base más una penalización:

$$\hat{\beta} = \arg\min_{\beta} \left[ \frac{1}{n}\|y - X\beta\|_2^2 + \lambda \cdot \text{Pen}(\beta) \right]$$

| Método | $\text{Pen}(\beta)$ | Efecto sobre coeficientes |
|--------|---------------------|---------------------------|
| OLS | — | Sin penalización; falla si $p \geq n$ |
| **Ridge** | $\sum_j \beta_j^2$ | Encoge todos hacia 0, nunca exactamente 0 |
| **Lasso** | $\sum_j |\beta_j|$ | Lleva muchos exactamente a 0 → selección de variables |
| **Elastic Net** | $\alpha\sum_j|\beta_j| + (1-\alpha)\sum_j\beta_j^2$ | Híbrido; `l1_ratio` $=\alpha$ controla el balance |

El $\lambda$ óptimo se elige por **validación cruzada** (5-fold por defecto en sklearn).

## 1. Importando ando

In [7]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import patsy
import warnings
import pyarrow
warnings.simplefilter('ignore')

from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(14052026)

## 2. Carga y exploración de datos

El archivo `salarios.csv` contiene observaciones de trabajadores con las siguientes variables:

| Variable | Descripción |
|----------|-------------|
| `lwage` | Log del salario por hora (**variable dependiente**) |
| `sex` | Sexo (1 = mujer) |
| `exp1` – `exp4` | Experiencia potencial y sus potencias (exp², exp³, exp⁴ / 100) |
| `shs` | Sin preparatoria completa |
| `hsg` | Preparatoria completa |
| `scl` | Algo de universidad |
| `clg` | Licenciatura |
| `ad` | Posgrado |
| `occ2` | Categoría de ocupación (2 niveles) |
| `ind2` | Categoría de industria (2 niveles) |
| `mw`, `so`, `we` | Región (Medio Oeste, Sur, Oeste; omitida = Noreste) |

In [8]:
# Cargar datos con Polars
df = pl.read_csv('salarios.csv')

print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print()
df.head()

Dimensiones: 5150 filas × 20 columnas



wage,lwage,sex,shs,hsg,scl,clg,ad,mw,so,we,ne,exp1,exp2,exp3,exp4,occ,occ2,ind,ind2
f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,f64,i64
9.615385,2.263364,1,0,0,0,1,0,0,0,0,1,7.0,0.49,0.343,0.2401,3600.0,11,8370.0,18
48.076923,3.872802,0,0,0,0,1,0,0,0,0,1,31.0,9.61,29.791,92.3521,3050.0,10,5070.0,9
11.057692,2.403126,0,0,1,0,0,0,0,0,0,1,18.0,3.24,5.832,10.4976,6260.0,19,770.0,4
13.942308,2.634928,1,0,0,0,0,1,0,0,0,1,25.0,6.25,15.625,39.0625,420.0,1,6990.0,12
28.846154,3.361977,1,0,0,0,1,0,0,0,0,1,22.0,4.84,10.648,23.4256,2015.0,6,9470.0,22


In [9]:
# Estadísticas descriptivas de las variables clave
df.select(['lwage', 'exp1', 'sex', 'shs', 'hsg', 'scl', 'clg', 'ad']).describe()

statistic,lwage,exp1,sex,shs,hsg,scl,clg,ad
str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",5150.0,5150.0,5150.0,5150.0,5150.0,5150.0,5150.0,5150.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",2.970787,13.760583,0.444466,0.023301,0.243883,0.278058,0.31767,0.137087
"""std""",0.570385,10.609465,0.496955,0.150872,0.429465,0.448086,0.465616,0.343973
"""min""",1.105912,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",2.599837,5.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",2.956512,10.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",3.324236,21.0,1.0,0.0,0.0,1.0,1.0,0.0
"""max""",6.270697,47.0,1.0,1.0,1.0,1.0,1.0,1.0


### Ejercicio — Exploración visual

Genera dos gráficas:
1. Histograma de `lwage`
2. Boxplot de `lwage` por nivel educativo (una caja para `shs`, `hsg`, `scl`, `clg`, `ad`)


In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

## 3. Preparación de matrices

Construimos dos versiones de la matriz de diseño $X$ usando `patsy`, que interpreta fórmulas estilo R:

**Modelo básico** — variables directas, sin interacciones:
```
lwage ~ sex + exp1 + exp2 + exp3 + exp4 + shs + hsg + scl + clg + C(occ2) + C(ind2) + mw + so + we
```

**Modelo con interacciones** — agrega los productos cruzados de cada `exp` con cada covariable de control:
```
lwage ~ sex + (exp1+exp2+exp3+exp4)*(shs+hsg+scl+clg+C(occ2)+C(ind2)+mw+so+we)
```
El operador `*` en patsy significa efectos principales + todas las interacciones de pares.
Esto genera como 400 columnas y ya no cabe con OLS, por eso nuestro objetivo es una matriz Z que contenga información relevante.

El `0 +` al inicio suprime el intercepto.

In [10]:
df_pd = df.to_pandas() # Porque patsy chilla si no es de pandas

y = df_pd['lwage'].values

# Modelo básico: variables directas
X_basic = patsy.dmatrix(
    '0 + sex + exp1 + exp2 + exp3 + exp4 + shs + hsg + scl + clg + C(occ2) + C(ind2) + mw + so + we',
    df_pd, return_type='dataframe'
)
feature_names_basic = X_basic.columns.tolist()
X_basic = X_basic.values

# Modelo con interacciones: exp × covariables
X_inter = patsy.dmatrix(
    '0 + sex + (exp1+exp2+exp3+exp4)*(shs+hsg+scl+clg+C(occ2)+C(ind2)+mw+so+we)',
    df_pd, return_type='dataframe'
)
feature_names_inter = X_inter.columns.tolist()
X_inter = X_inter.values

print(f'X básica:          {X_basic.shape[0]} obs × {X_basic.shape[1]} variables')
print(f'X con interacciones: {X_inter.shape[0]} obs × {X_inter.shape[1]} variables')

X básica:          5150 obs × 54 variables
X con interacciones: 5150 obs × 246 variables


### División train / test

Separamos 20% de los datos como conjunto de test. **Ambas matrices comparten el mismo split** para que las comparaciones sean justas.

In [11]:
# Split común para ambas matrices
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)

X_basic_train, X_basic_test   = X_basic[idx_train], X_basic[idx_test]
X_inter_train, X_inter_test   = X_inter[idx_train], X_inter[idx_test]
y_train, y_test               = y[idx_train],        y[idx_test]

n_train = len(y_train)
print(f'Train: {n_train} obs  |  Test: {len(y_test)} obs')
print(f'Modelo básico:      p = {X_basic_train.shape[1]}  p/n = {X_basic_train.shape[1]/n_train:.2f}')
print(f'Con interacciones:  p = {X_inter_train.shape[1]}  p/n = {X_inter_train.shape[1]/n_train:.2f}')

Train: 4120 obs  |  Test: 1030 obs
Modelo básico:      p = 54  p/n = 0.01
Con interacciones:  p = 246  p/n = 0.06


---
## 4. Parte 1 — Modelo básico con OLS

Con $p \ll n$ la regresión OLS es perfectamente válida. Usamos `LinearRegression` de sklearn como benchmar antes de agregar interacciones.

### Ejercicio Ajustar OLS y calcular métricas

1. Ajusta `LinearRegression` sobre `(X_basic_train, y_train)`
2. Calcula $R^2$ y $\sqrt{MSE}$ (RMSE) en **train** y en **test**
3. Construye un `pl.DataFrame` con los resultados


In [6]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────

# Ajustar OLS
ols = 

# Métricas
r2_ols_train  =
r2_ols_test   = 
rmse_ols_train = 
rmse_ols_test  = 

# Tabla Polars
resultados_ols = pl.DataFrame({
    'Modelo':  ['OLS básico'],
    'R² Train': [],
    'R² Test':  [],
    'RMSE Train': [],
    'RMSE Test':  [],
})
print(resultados_ols)

SyntaxError: invalid syntax (1194960424.py, line 4)

### Ejercicio Gráfico de residuales

Genera un scatter de valores ajustados vs. residuales en el conjunto de test. 

Añade también una línea horizontal en $y=0$ como referencia.

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

### Ejercicio coeficientes OLS

Usa Polars para construir un DataFrame con los nombres de las variables y sus coeficientes, ordenado de mayor a menor por valor absoluto. Muestra los 10 más influyentes.


In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

---
## 5. Modelo con interacciones y regresiones penalizadas

Ahora trabajamos con `X_inter` que tiene $p \gg n_{train}$ y OLS chilla. Los métodos penalizados resuelven esto encogiendo o eliminando coeficientes.

### Ejercicio Ajustar Lasso, Ridge y Elastic Net

Entrena los tres modelos sobre `(X_inter_train, y_train)`. El $\lambda$ se elige automáticamente por validación cruzada.

Calcula $R^2$ y RMSE en **test** para los tres métodos **y para OLS básico** (cópialo de la sección anterior para comparar).

> **Parámetros sugeridos:**
> - `LassoCV(cv=5, max_iter=10000)`
> - `RidgeCV(alphas=np.logspace(-3, 4, 50))`
> - `ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95], cv=5, max_iter=10000)`

In [12]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────

# Entrenar modelos penalizados
lasso = 
ridge = 
enet  = 

# Métricas en test
resultados = pl.DataFrame({
    'Modelo':     ['OLS básico', 'Lasso', 'Ridge', 'Elastic Net'],
    'p':          [X_basic_train.shape[1], X_inter_train.shape[1],
                   X_inter_train.shape[1], X_inter_train.shape[1]],
    'R² Test':    [r2_ols_test, , , ],
    'RMSE Test':  [rmse_ols_test, , , ],
})
print(resultados)

SyntaxError: invalid syntax (2098041734.py, line 4)

### Ejercicio — Penalizaciones seleccionadas

Imprime el $\lambda$ óptimo elegido por validación cruzada para cada modelo.

**Pista:** `lasso.alpha_`, `ridge.alpha_`, `enet.alpha_` y `enet.l1_ratio_`

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

### Ejercicio Sparsidad de Lasso

Una ventaja clave de Lasso es la selección automática de variables: lleva los coeficientes irrelevantes exactamente a cero.

1. Cuenta cuántos coeficientes de Lasso son exactamente cero y cuántos son distintos de cero
2. Construye un `pl.DataFrame` con las variables seleccionadas (coef != 0) y sus coeficientes, ordenado por magnitud
3. Grafica los coeficientes de Lasso vs. Ridge en un scatter (un punto por variable). ¿Qué patrón observas?

**Pista:** `lasso.coef_` y `ridge.coef_` tienen longitud `p`

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

---
## 6. Visualización comparativa

### Ejercicio Predicho vs. real

Genera un panel de 4 subplots (2×2): uno por modelo (OLS básico, Lasso, Ridge, Elastic Net).
En cada subplot grafica $y_{test}$ en el eje X vs. $\hat{y}_{test}$ en el eje Y, con la línea de 45° como referencia.
Incluye el $R^2$ en el título de cada subplot.

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

### Ejercicio Gráfico de barras comparativo

Usando el data.frame de resultados, genera un gráfico de barras horizontales con el $R^2$ Test de los cuatro modelos, ordenado de mayor a menor.

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

---
## 7. Preguntas de reflexión

Responde

¿Por qué OLS sobre el modelo con interacciones no es factible?

*Tu respuesta aquí:*

Al comparar OLS básico con los modelos penalizados sobre interacciones, ¿cuál tiene mejor $R^2$ en test? ¿Qué implica eso sobre el rol de las interacciones?

*Tu respuesta aquí:*

Lasso seleccionó un subconjunto de las variables. ¿Qué tipo de interacciones conservó? ¿Tiene sentido económico?

*Tu respuesta aquí:*

¿Por qué NO deberías usar los coeficientes de Lasso para hacer inferencia causal sobre el efecto del sexo en el salario? ¿Qué propiedad de Lasso lo hace inadecuado para eso?

*Tu respuesta aquí:*

---
## 8. Análisis de la brecha salarial por sexo

### Brecha sin controles

Calcula la diferencia cruda en `lwage` promedio entre hombres y mujeres usando Polars:
```python
df.group_by('sex').agg(pl.col('lwage').mean())
```
¿Cuánto es la brecha en puntos de log-salario? ¿Cómo se interpreta ese número?

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────



# ──────────────────────────────────────────────────────────────────────────────

### Brecha controlada por OLS

El coeficiente de `sex` en la regresión OLS básica es la brecha salarial controlando por educación, experiencia, ocupación, industria y región. Extráelo y compáralo con la brecha cruda.

¿Qué fracción de la brecha cruda se explica por diferencias en características observables?

In [ ]:
# ── TU CÓDIGO AQUÍ ────────────────────────────────────────────────────────────
# Pista: el índice de 'sex' en feature_names_basic es feature_names_basic.index('sex')



# ──────────────────────────────────────────────────────────────────────────────